# 03 — Fine-Tune Agent 2: Secure SDLC Architect (`qwen2.5-3b-cc-agent2`)

Fine-tunes `unsloth/Qwen2.5-3B-Instruct` (4-bit QLoRA, `r=16`, `lora_alpha=16`) to weigh extracted security requirements, identify the dominant constraint, and recommend an optimal SDLC with phase mapping.

In [ ]:
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers<5.0.0" "trl<=1.13.0" "datasets<4.4.0" "unsloth-zoo>=2026.9.0"

In [ ]:
# 0. Ensure stable dependencies are loaded
!pip install -q "transformers<5.0.0" "trl<=1.13.0" "datasets<4.4.0" "unsloth-zoo>=2026.9.0"

# 1. MUST BE THE VERY FIRST IMPORT to apply GPU optimizations
import unsloth
import json
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template

# 2. Agent 2 Dataset with explicit Requirement Weighting
dataset_content = [
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall resist physical manipulation and probing of the TPM hardware."},
      {"req_id": "REQ-002", "statement": "The system shall run self-tests at initialisation."},
      {"req_id": "REQ-003", "statement": "The system shall generate audit logs for startup and shutdown."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "High", "pulls_toward": "V-Model", "reason": "Physical hardware constraints are immutable post-manufacture."},
        {"req_id": "REQ-002", "weight": "Medium", "pulls_toward": "Hardware Lifecycle", "reason": "Low-level boot logic requires strict upfront verification."},
        {"req_id": "REQ-003", "weight": "Low", "pulls_toward": "Agnostic", "reason": "Logging is standard software logic, easily patched."}
      ],
      "dominant_constraint": "Hardware immutability (REQ-001) heavily outweighs software logging (REQ-003). Mistakes cannot be patched via OTA.",
      "recommended_sdlc": "High-Assurance V-Model",
      "justification": "Because physical hardware (TPM) cannot be modified after deployment, the left side of the V-Model ensures mathematical and physical constraints are proven before manufacturing begins. Agile approaches are incompatible with physical silicon fabrication.",
      "phase_mapping": [
        {"phase": "Hardware Requirements Analysis", "applicable_reqs": ["REQ-001"]},
        {"phase": "Integration & Silicon Testing", "applicable_reqs": ["REQ-002", "REQ-003"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall inspect network traffic against dynamic signature baselines."},
      {"req_id": "REQ-002", "statement": "The system shall drop malicious traffic flows in real time."},
      {"req_id": "REQ-003", "statement": "The system shall update anomaly thresholds based on live telemetry."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "High", "pulls_toward": "DevSecOps", "reason": "Dynamic signatures require continuous integration of threat feeds."},
        {"req_id": "REQ-002", "weight": "Medium", "pulls_toward": "Agile", "reason": "Real-time blocking requires fast iterative deployment of rules."},
        {"req_id": "REQ-003", "weight": "High", "pulls_toward": "DevSecOps", "reason": "Live telemetry feedback loops necessitate continuous deployment (CI/CD)."}
      ],
      "dominant_constraint": "High volatility of the threat landscape requires continuous deployment and automated rule testing.",
      "recommended_sdlc": "DevSecOps (Continuous Agile)",
      "justification": "An IPS dealing with live threat intelligence cannot wait for a 6-month Waterfall release cycle. DevSecOps allows continuous automated testing (DAST/SAST) and daily deployments of new signature baselines.",
      "phase_mapping": [
        {"phase": "Continuous Integration (CI)", "applicable_reqs": ["REQ-001", "REQ-003"]},
        {"phase": "Continuous Deployment (CD) & Monitor", "applicable_reqs": ["REQ-002"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The TSF shall provide a means to verify firmware updates using a digital signature before installation."},
      {"req_id": "REQ-002", "statement": "The TSF shall prevent unauthorized administrators from initiating updates."},
      {"req_id": "REQ-003", "statement": "The TSF shall maintain a secure execution domain for the update logic."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "High", "pulls_toward": "Secure Spiral", "reason": "Firmware updates blend static hardware boundaries with iterative software patching."},
        {"req_id": "REQ-002", "weight": "Medium", "pulls_toward": "Agnostic", "reason": "RBAC logic applies to any lifecycle."},
        {"req_id": "REQ-003", "weight": "High", "pulls_toward": "V-Model", "reason": "Secure domain separation requires rigorous upfront architectural design."}
      ],
      "dominant_constraint": "The system must balance rigid upfront architectural design (Secure Domain) with the ability to safely iterate software post-deployment (OTA Updates).",
      "recommended_sdlc": "Secure Spiral Model",
      "justification": "The Secure Spiral model is ideal for firmware. The initial hardware and cryptographic boundaries (REQ-003) are established rigidly in the first spirals, while the digital signature mechanism (REQ-001) enables subsequent spirals to be delivered securely over the air.",
      "phase_mapping": [
        {"phase": "Architectural Risk Analysis (Early Spirals)", "applicable_reqs": ["REQ-003"]},
        {"phase": "Iterative Release & Verify (Later Spirals)", "applicable_reqs": ["REQ-001", "REQ-002"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall enforce a Mandatory Access Control (MAC) policy on all OS objects."},
      {"req_id": "REQ-002", "statement": "The system shall provide a mathematical proof that the access control policy cannot be bypassed."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "Medium", "pulls_toward": "Agnostic", "reason": "Standard OS-level access control."},
        {"req_id": "REQ-002", "weight": "Critical", "pulls_toward": "Formal Methods / Cleanroom", "reason": "Mathematical proofs of non-bypassability require strict formal specifications."}
      ],
      "dominant_constraint": "The requirement for mathematical proof of non-bypassability eliminates fast-iteration methodologies.",
      "recommended_sdlc": "Cleanroom Software Engineering",
      "justification": "When formal mathematical proofs of security (Common Criteria EAL5+) are required, Cleanroom Engineering must be used. It relies on formal specification, statistical testing, and zero-defect tolerances rather than iterative debugging.",
      "phase_mapping": [
        {"phase": "Formal Specification", "applicable_reqs": ["REQ-001", "REQ-002"]},
        {"phase": "Statistical Testing", "applicable_reqs": ["REQ-002"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall support password authentication plus MFA."},
      {"req_id": "REQ-002", "statement": "The system shall terminate inactive sessions after 15 minutes."},
      {"req_id": "REQ-003", "statement": "The system shall lock accounts after 5 failed login attempts."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "Low", "pulls_toward": "Agile", "reason": "Standard web application authentication logic."},
        {"req_id": "REQ-002", "weight": "Low", "pulls_toward": "Agile", "reason": "Simple session state management."},
        {"req_id": "REQ-003", "weight": "Low", "pulls_toward": "Agile", "reason": "Basic business logic implementation."}
      ],
      "dominant_constraint": "No hardware, formal proof, or high-assurance constraints exist. Speed to market and usability govern.",
      "recommended_sdlc": "Secure Scrum / Agile",
      "justification": "These are standard web identity requirements. Secure Agile (Scrum with integrated security user stories) allows fast implementation of MFA and session timeouts while adapting to changing UI/UX needs.",
      "phase_mapping": [
        {"phase": "Sprint Planning (Security User Stories)", "applicable_reqs": ["REQ-001", "REQ-002", "REQ-003"]},
        {"phase": "Sprint Review (Penetration Testing)", "applicable_reqs": ["REQ-001"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The initialisation function shall verify authenticity of the bootloader using fused hardware keys."},
      {"req_id": "REQ-002", "statement": "If authenticity fails, the MCU shall halt execution immediately."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "High", "pulls_toward": "V-Model", "reason": "Fused hardware keys are permanently burned into silicon."},
        {"req_id": "REQ-002", "weight": "High", "pulls_toward": "V-Model", "reason": "Halting execution at boot requires precise low-level hardware interrupts."}
      ],
      "dominant_constraint": "Silicon immutability (fused keys) dictates a highly rigid verification pipeline.",
      "recommended_sdlc": "Hardware V-Model",
      "justification": "Because keys are fused into the Microcontroller Unit (MCU) at the factory, any error bricks the device permanently. The V-Model's emphasis on exhaustive testing before the 'Implementation' (fabrication) phase is mandatory.",
      "phase_mapping": [
        {"phase": "System Design & Key Architecture", "applicable_reqs": ["REQ-001"]},
        {"phase": "Hardware-in-the-Loop (HIL) Testing", "applicable_reqs": ["REQ-002"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall implement TLS 1.3 and reject all older versions."},
      {"req_id": "REQ-002", "statement": "The system shall support mutual authentication using X.509v3 certificates."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "Medium", "pulls_toward": "Agile", "reason": "Network protocol configurations are easily updated via software patches."},
        {"req_id": "REQ-002", "weight": "Medium", "pulls_toward": "Agile", "reason": "Certificate parsing is standard software library functionality."}
      ],
      "dominant_constraint": "Software-defined network configurations allow for rapid iteration without hardware penalties.",
      "recommended_sdlc": "Secure Agile",
      "justification": "TLS versions and ciphersuites become deprecated as new vulnerabilities are found. An Agile approach allows the development team to quickly patch configurations and roll out new certificate requirements without heavy architectural redesigns.",
      "phase_mapping": [
        {"phase": "Implementation (Sprint Execution)", "applicable_reqs": ["REQ-001", "REQ-002"]},
        {"phase": "Automated Security Testing (DAST)", "applicable_reqs": ["REQ-001"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall encrypt data-at-rest using AES-256-GCM."},
      {"req_id": "REQ-002", "statement": "The system shall destroy cryptographic keys from memory immediately after use."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "Medium", "pulls_toward": "Waterfall / Spiral", "reason": "Core database schemas and encryption are difficult to alter once data is ingested."},
        {"req_id": "REQ-002", "weight": "Medium", "pulls_toward": "Waterfall / Spiral", "reason": "Memory management requires careful upfront architectural planning."}
      ],
      "dominant_constraint": "Data gravity: Once petabytes of data are encrypted, changing the encryption scheme is prohibitively expensive.",
      "recommended_sdlc": "Iterative Waterfall",
      "justification": "While not as rigid as hardware, database encryption requires careful upfront planning (Waterfall) because migrating encrypted data later is highly disruptive. Iterations are allowed, but the core cryptographic architecture must be frozen early.",
      "phase_mapping": [
        {"phase": "System Architecture & Design", "applicable_reqs": ["REQ-001", "REQ-002"]},
        {"phase": "Unit & Integration Testing", "applicable_reqs": ["REQ-002"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The firewall shall enforce policy based on IP, port, and application type."},
      {"req_id": "REQ-002", "statement": "The firewall shall ensure policy enforcement cannot be bypassed by malformed packets."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "Low", "pulls_toward": "DevSecOps", "reason": "Standard packet filtering logic."},
        {"req_id": "REQ-002", "weight": "High", "pulls_toward": "DevSecOps", "reason": "Malformed packet defenses require continuous fuzzing and rapid deployment."}
      ],
      "dominant_constraint": "Network threat actors constantly discover new malformed packet bypasses, necessitating rapid response.",
      "recommended_sdlc": "DevSecOps",
      "justification": "Firewalls are software-defined and face evolving threats. DevSecOps allows developers to integrate continuous fuzzing into the CI/CD pipeline, ensuring malformed packet bypasses are caught automatically during every build.",
      "phase_mapping": [
        {"phase": "Automated Testing (Fuzzing)", "applicable_reqs": ["REQ-002"]},
        {"phase": "Deployment", "applicable_reqs": ["REQ-001"]}
      ]
    }
  },
  {
    "input_requirements": [
      {"req_id": "REQ-001", "statement": "The system shall restrict gateway management to the authorized administrator role."},
      {"req_id": "REQ-002", "statement": "The system shall maintain a distinct security domain isolated from untrusted interference."}
    ],
    "output_sdlc": {
      "requirements_weighting": [
        {"req_id": "REQ-001", "weight": "Low", "pulls_toward": "Agnostic", "reason": "Standard RBAC administration."},
        {"req_id": "REQ-002", "weight": "High", "pulls_toward": "Spiral", "reason": "Security domain separation requires strong architectural foundation but allows software iteration."}
      ],
      "dominant_constraint": "Architectural isolation must be established first, but management features can evolve.",
      "recommended_sdlc": "Secure Spiral Model",
      "justification": "The Spiral model supports risk-driven development. The highest risk—domain separation (REQ-002)—is addressed in the first architectural spiral. Subsequent spirals build out the administrative logic and RBAC controls (REQ-001).",
      "phase_mapping": [
        {"phase": "Spiral 1: Core Architecture", "applicable_reqs": ["REQ-002"]},
        {"phase": "Spiral 2: Feature Development", "applicable_reqs": ["REQ-001"]}
      ]
    }
  }
]

with open("agent2_training_dataset.json", "w") as f:
    json.dump(dataset_content, f)

# 3. Load Base Model & Tokenizer for Agent 2
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# 4. Agent 2 Specific Formatting (Mapping Schema to ChatML)
def format_qwen_agent2(example):
    user_prompt = "Requirements:\n" + json.dumps(example['input_requirements'], indent=2)
    assistant_response = json.dumps(example['output_sdlc'], indent=2)

    messages = [
        {"role": "system", "content": "You are a Secure SDLC Architect. Evaluate the provided requirements, assign a weight/pull factor to each to determine the dominant constraint, and output a JSON object recommending the optimal SDLC with justification."},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_response}
    ]
    return {"messages": messages}

raw_dataset = Dataset.from_json("agent2_training_dataset.json")
chat_dataset = raw_dataset.map(format_qwen_agent2)

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def apply_template(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

train_dataset = chat_dataset.map(apply_template, batched=True)

# 5. Execute Training Loop for Agent 2
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 2048,
        dataset_num_proc = 2,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "agent2_outputs",
    ),
)

trainer.train()

## Save Adapter & Evaluate on Agent 1 Firmware Output

In [ ]:
import json
from google.colab import files

# 1. Save Agent 2 adapters locally
model.save_pretrained("qwen2.5-3b-cc-agent2")
tokenizer.save_pretrained("qwen2.5-3b-cc-agent2")

# 2. Create zip backup and download
!zip -r qwen2.5-3b-cc-agent2.zip qwen2.5-3b-cc-agent2/
files.download("qwen2.5-3b-cc-agent2.zip")

# 3. Enable fast inference mode
FastLanguageModel.for_inference(model)

# 4. Test Agent 2 on the exact output generated by Agent 1 earlier
agent1_firmware_output = [
  {
    "category": "Security - Integrity",
    "reasoning": "Without this requirement, an attacker could replace the firmware on a device without leaving any trace...",
    "req_id": "REQ-001",
    "source_ref": "FPT_TUD_EXT.1.3 (Trusted Update)",
    "statement": "The TSF shall provide a means to verify firmware/software updates to the TOE using a digital signature mechanism prior to installing those updates."
  },
  {
    "category": "Security - Integrity",
    "reasoning": "Without this requirement, an attacker could install unauthorized firmware updates...",
    "req_id": "REQ-002",
    "source_ref": "FPT_TUD_EXT.1.2 (Trusted Update)",
    "statement": "The TSF shall provide the ability to initiate updates to TOE firmware/software."
  },
  {
    "category": "Security - Integrity",
    "reasoning": "Without this requirement, an attacker could replace the firmware update process...",
    "req_id": "REQ-003",
    "source_ref": "FPT_TUD_EXT.1.1 (Trusted Update)",
    "statement": "The TSF shall provide the ability to query the current version of the TOE firmware/software."
  }
]

eval_messages = [
    {"role": "system", "content": "You are a Secure SDLC Architect. Evaluate the provided requirements, assign a weight/pull factor to each to determine the dominant constraint, and output a JSON object recommending the optimal SDLC with justification."},
    {"role": "user", "content": "Requirements:\n" + json.dumps(agent1_firmware_output, indent=2)}
]

inputs = tokenizer.apply_chat_template(
    eval_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=1024, use_cache=True)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print(response)